In [ ]:
# Colab / A100: FAST EVIDENCE MODE for ΔF(Σ,a) with a RESCALED (fixed-physical-size) sheet
# - 3 t-points: (0, 0.5, 1)
# - tiny stats (fast trend check)
# - sheet is codim-2 AND spans a fixed FRACTION of the box in the two free directions
#   (so physical area stays fixed if you interpret a ~ 1/N with fixed physical box size)

import math
import torch

# -----------------------------
# Device / dtype
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
cdtype = torch.complex64
rdtype = torch.float32
print("device:", device)

# -----------------------------
# SU(3) utilities
# -----------------------------
def proj_su3(U: torch.Tensor) -> torch.Tensor:
    """Project (...,3,3) complex matrices to SU(3) via QR + det correction."""
    Q, R = torch.linalg.qr(U)
    diag = torch.diagonal(R, dim1=-2, dim2=-1)
    phase = diag / (diag.abs() + 1e-12)
    Q = Q * phase.conj().unsqueeze(-2)
    detQ = torch.linalg.det(Q)
    det_root = detQ ** (1/3)
    Q = Q / det_root.unsqueeze(-1).unsqueeze(-1)
    return Q

def random_su3(shape, sigma=0.20, device=device) -> torch.Tensor:
    """Near-identity random SU(3) draws using exp(iH) and projection."""
    A = torch.randn(*shape, 3, 3, device=device, dtype=cdtype) * sigma
    H = (A + A.conj().transpose(-2, -1)) * 0.5
    tr = torch.diagonal(H, dim1=-2, dim2=-1).sum(-1, keepdim=True)
    H = H - (tr / 3.0).unsqueeze(-1) * torch.eye(3, device=device, dtype=cdtype)
    U = torch.linalg.matrix_exp(torch.tensor(1j, device=device, dtype=cdtype) * H)
    return proj_su3(U)

def su3_dag(U): return U.conj().transpose(-2, -1)
def roll(U, shift, dim): return torch.roll(U, shifts=shift, dims=dim)

# -----------------------------
# Lattice objects
# -----------------------------
# Links: U[t,z,y,x,mu,3,3], mu=0,1,2,3 corresponds to +t,+z,+y,+x
def plaquette(U, mu, nu):
    """Oriented plaquette U_mu(x) U_nu(x+mu) U_mu(x+nu)† U_nu(x)†."""
    U_mu = U[..., mu, :, :]
    U_nu = U[..., nu, :, :]
    U_nu_xpmu = roll(U_nu, -1, mu)
    U_mu_xpnu = roll(U_mu, -1, nu)
    return U_mu @ U_nu_xpmu @ su3_dag(U_mu_xpnu) @ su3_dag(U_nu)

def local_staple(U, mu):
    """Staple sum for Wilson action for links in direction mu."""
    Nt, Nz, Ny, Nx, _, _, _ = U.shape
    staple = torch.zeros((Nt, Nz, Ny, Nx, 3, 3), device=U.device, dtype=U.dtype)
    for nu in range(4):
        if nu == mu:
            continue
        U_mu = U[..., mu, :, :]
        U_nu = U[..., nu, :, :]

        # forward staple: U_nu(x) U_mu(x+nu) U_nu(x+mu)†
        term_f = U_nu @ roll(U_mu, -1, nu) @ su3_dag(roll(U_nu, -1, mu))

        # backward staple: U_nu(x-nu)† U_mu(x-nu) U_nu(x-nu+mu)
        U_nu_xmnu = roll(U_nu, +1, nu)
        term_b = su3_dag(U_nu_xmnu) @ roll(U_mu, +1, nu) @ roll(U_nu_xmnu, -1, mu)

        staple = staple + term_f + term_b
    return staple

# -----------------------------
# RESCALED (fixed-physical-size) sheet mask
# -----------------------------
def make_sheet_mask_fixed_physical(
    Nt, Nz, Ny, Nx,
    fixed_dims=(1, 2),          # fix (z,y) by default
    fixed_indices=(0, 0),
    span_fracs=(1.0, 1.0),      # spans full remaining two dims by default
    span_offsets=(0.0, 0.0),    # where the span starts (fractions)
    device=device,
):
    """
    Codim-2 sheet mask on plaquette basepoints (Nt,Nz,Ny,Nx).
    Fixes two coordinates and spans a window in the other two coordinates.
    Window size scales with N via span_fracs => fixed physical size if a ~ 1/N with fixed box size.
    """
    sizes = [Nt, Nz, Ny, Nx]
    all_dims = [0, 1, 2, 3]
    free_dims = [d for d in all_dims if d not in fixed_dims]
    assert len(free_dims) == 2, "Need exactly 2 fixed dims for codim-2 sheet."

    coords = torch.stack(torch.meshgrid(
        torch.arange(Nt, device=device),
        torch.arange(Nz, device=device),
        torch.arange(Ny, device=device),
        torch.arange(Nx, device=device),
        indexing="ij"
    ), dim=-1)

    m = torch.ones((Nt, Nz, Ny, Nx), device=device, dtype=torch.bool)

    for d, idx in zip(fixed_dims, fixed_indices):
        m &= (coords[..., d] == (int(idx) % sizes[d]))

    for k, d in enumerate(free_dims):
        L = sizes[d]
        span = max(1, int(round(span_fracs[k] * L)))
        start = int(round(span_offsets[k] * L)) % L
        idxs = (coords[..., d] - start) % L
        m &= (idxs < span)

    return m

def tHooft_delta_sum(U, beta, z_phase, mask, mu_sheet, nu_sheet):
    """
    O(U)= Σ_{p in sheet} Δ_p(U), Δ_p = (beta/3)( ReTr(U_p) - ReTr(z U_p) ).
    """
    P = plaquette(U, mu_sheet, nu_sheet)
    trP = torch.diagonal(P, dim1=-2, dim2=-1).sum(-1)  # complex trace
    re_trP = torch.real(trP)
    re_tr_zP = torch.real(z_phase * trP)
    Delta = (beta / 3.0) * (re_trP - re_tr_zP)
    return torch.sum(Delta[mask])

# -----------------------------
# Metropolis sweep (fast evidence mode)
# Correct for bulk Wilson part; twist part handled by recomputing sheet plaquettes before/after each (parity,mu) batch.
# -----------------------------
@torch.no_grad()
def metropolis_sweep_fast(U, beta, t, z_phase, mask, mu_sheet, nu_sheet, step_sigma=0.18):
    Nt, Nz, Ny, Nx, _, _, _ = U.shape

    coords = torch.stack(torch.meshgrid(
        torch.arange(Nt, device=U.device),
        torch.arange(Nz, device=U.device),
        torch.arange(Ny, device=U.device),
        torch.arange(Nx, device=U.device),
        indexing="ij"
    ), dim=-1)
    parity = (coords.sum(-1) & 1).to(torch.bool)

    # Precompute sheet plaquette contribution field if twist is active and relevant
    for par in [0, 1]:
        sel = (~parity) if par == 0 else parity

        for mu in range(4):
            staple = local_staple(U, mu)
            U_mu = U[..., mu, :, :]

            U_sel = U_mu[sel]
            staple_sel = staple[sel]

            R = random_su3((U_sel.shape[0],), sigma=step_sigma, device=U.device)
            U_prop = proj_su3(R @ U_sel)

            stap_dag = su3_dag(staple_sel)
            old_val = torch.real(torch.einsum("bij,bji->b", U_sel, stap_dag))
            new_val = torch.real(torch.einsum("bij,bji->b", U_prop, stap_dag))
            dS = -(beta / 3.0) * (new_val - old_val)

            if t != 0.0 and (mu == mu_sheet or mu == nu_sheet):
                # sheet plaquette field before
                P_before = plaquette(U, mu_sheet, nu_sheet)
                tr_b = torch.diagonal(P_before, dim1=-2, dim2=-1).sum(-1)
                d_b = (beta/3.0) * (torch.real(tr_b) - torch.real(z_phase * tr_b))

                # apply batch proposal
                U_tmp = U.clone()
                U_tmp[..., mu, :, :][sel] = U_prop

                # sheet plaquette field after
                P_after = plaquette(U_tmp, mu_sheet, nu_sheet)
                tr_a = torch.diagonal(P_after, dim1=-2, dim2=-1).sum(-1)
                d_a = (beta/3.0) * (torch.real(tr_a) - torch.real(z_phase * tr_a))

                dd = d_a - d_b  # per-plaquette-basepoint change

                other = nu_sheet if mu == mu_sheet else mu_sheet
                dd_shift = roll(dd, +1, other)
                mask_shift = roll(mask, +1, other)

                twist_dS = (dd[sel] * mask[sel].to(dd.dtype)) + (dd_shift[sel] * mask_shift[sel].to(dd.dtype))
                dS = dS + t * twist_dS

            u = torch.rand_like(dS, dtype=rdtype, device=U.device)
            accept = (dS <= 0) | (torch.exp(-dS) > u)

            U_new = torch.where(accept.view(-1, 1, 1), U_prop, U_sel)
            U_mu[sel] = U_new
            U[..., mu, :, :] = proj_su3(U_mu)

    return U

# -----------------------------
# Thermodynamic integration: ΔF = ∫_0^1 <O_t>_t dt (evidence mode: 3 t points)
# -----------------------------
@torch.no_grad()
def quick_deltaF(N=8, beta=6.0, seed=1,
                 t_grid=(0.0, 0.5, 1.0),
                 therm0=20, therm_each=10, meas=20, thin=1,
                 step_sigma=0.18,
                 mu_sheet=0, nu_sheet=3,
                 # rescaled sheet parameters:
                 fixed_dims=(1,2), fixed_indices=(0,0),        # fix z=0,y=0
                 span_fracs=(0.5, 0.5), span_offsets=(0.0,0.0),# span 1/2 box in t and x (fixed physical area fraction)
                 z_k=1):
    torch.manual_seed(seed)
    Nt = Nz = Ny = Nx = N

    omega = torch.exp(torch.tensor(2j * math.pi / 3, device=device, dtype=cdtype))
    z_phase = omega ** z_k

    mask = make_sheet_mask_fixed_physical(
        Nt, Nz, Ny, Nx,
        fixed_dims=fixed_dims,
        fixed_indices=fixed_indices,
        span_fracs=span_fracs,
        span_offsets=span_offsets,
        device=device
    )

    U = random_su3((Nt, Nz, Ny, Nx, 4), sigma=0.15, device=device)

    for _ in range(therm0):
        U = metropolis_sweep_fast(U, beta, t_grid[0], z_phase, mask, mu_sheet, nu_sheet, step_sigma)

    O_means = []
    for t in t_grid:
        for _ in range(therm_each):
            U = metropolis_sweep_fast(U, beta, t, z_phase, mask, mu_sheet, nu_sheet, step_sigma)

        vals = []
        for _ in range(meas):
            for __ in range(thin):
                U = metropolis_sweep_fast(U, beta, t, z_phase, mask, mu_sheet, nu_sheet, step_sigma)
            vals.append(tHooft_delta_sum(U, beta, z_phase, mask, mu_sheet, nu_sheet).item())

        v = torch.tensor(vals, dtype=torch.float64)
        m = v.mean().item()
        s = v.std(unbiased=True).item() if len(vals) > 1 else 0.0
        O_means.append(m)
        print(f"t={t:4.2f}  <O_t>={m:+.6e}  std={s:.3e}")

    DeltaF = 0.0
    for i in range(len(t_grid) - 1):
        dt = t_grid[i + 1] - t_grid[i]
        DeltaF += 0.5 * dt * (O_means[i] + O_means[i + 1])

    print(f"\nΔF ≈ {DeltaF:.6e}   <V_Σ>≈exp(-ΔF)")
    return DeltaF, list(t_grid), O_means

# -----------------------------
# RUN: N sweep (rescaled sheet)
# IMPORTANT: span_fracs fixed -> sheet area scales like N^2 in lattice units, fixed fraction in physical units.
# -----------------------------
for N in [6, 8, 10]:
    print("\n============================")
    print(f"RUN N={N}")
    print("============================")
    DeltaF, t_grid, O_means = quick_deltaF(
        N=N, beta=6.0, seed=1,
        t_grid=(0.0, 0.5, 1.0),
        therm0=20, therm_each=10, meas=20, thin=1,
        step_sigma=0.18,
        mu_sheet=0, nu_sheet=3,          # (t,x) plaquettes
        fixed_dims=(1,2), fixed_indices=(0,0),   # z=0,y=0
        span_fracs=(0.5, 0.5), span_offsets=(0.0,0.0),  # half-box in t and x
        z_k=1
    )
    print(f"N={N}  ΔF≈{DeltaF:.6e}  <VΣ>≈exp(-ΔF)")

device: cuda

RUN N=6
t=0.00  <O_t>=+5.109295e+01  std=2.762e+00
t=0.50  <O_t>=+3.167217e+01  std=2.572e+00
t=1.00  <O_t>=+4.189533e+00  std=2.954e+00

ΔF ≈ 2.965670e+01   <V_Σ>≈exp(-ΔF)
N=6  ΔF≈2.965670e+01  <VΣ>≈exp(-ΔF)

RUN N=8
t=0.00  <O_t>=+9.874235e+01  std=2.453e+00
t=0.50  <O_t>=+6.804955e+01  std=6.771e+00
t=1.00  <O_t>=+1.960360e+01  std=1.604e+01

ΔF ≈ 6.361126e+01   <V_Σ>≈exp(-ΔF)
N=8  ΔF≈6.361126e+01  <VΣ>≈exp(-ΔF)

RUN N=10
t=0.00  <O_t>=+1.440285e+02  std=4.258e+00
t=0.50  <O_t>=+1.043678e+02  std=1.070e+01
t=1.00  <O_t>=+2.248372e+01  std=1.642e+01

ΔF ≈ 9.381194e+01   <V_Σ>≈exp(-ΔF)
N=10  ΔF≈9.381194e+01  <VΣ>≈exp(-ΔF)
